# Demo 03: Verify and Repair

Week 4, Module 01. Reliability, hallucination risk, and grounding.

Instructor demo (we-do). This is the third step in the Week 4 reliability arc. Demo 2 built a grounded assistant that cites its sources and abstains. This demo adds the next layer: force the answer into a checkable structure, run an automated verifier that catches unsupported claims, and repair the answer in a loop until it passes. Then we log attempts and cost so we can talk about the trade-off out loud.

Where you see a WE-DO banner in a cell, that is a line we write together as a room. Everything else is pre-filled.

## The retail hook

The Cordwell assistant is about to be wired into a workflow, not just read by a person. That means its output has to be machine-checkable. We make it emit JSON with an answer, citations, a confidence, and an abstained flag. Then a verifier checks the claims against the evidence before anything downstream trusts them. When a check fails, a repair step rewrites the answer. This is the difference between hoping a model is faithful and enforcing it.

## Engineer framing

Three ideas map onto tools you already use.

- A schema guard is input validation for model output. Same idea as validating a request body against a type before you process it.
- A verifier is a test suite for a single answer. It runs fast assertions against the evidence and returns the failures.
- A repair loop is retry-with-feedback. On failure you feed the specific issues back in and try again, up to a cap.

## What you will be able to do

1. Force model output into a Pydantic schema and reject anything that does not fit.
2. Write a heuristic verifier that checks quote containment, inline citations, and numeric and year consistency against the evidence.
3. Drive a verify-then-repair loop that fixes an unfaithful answer automatically.
4. Add an abstention policy that declines on low confidence.
5. Log attempts, tokens, and estimated cost, and read the reliability-versus-cost trade-off off a chart.

Time budget: about 55 minutes. Natural break after the counterexample trap in Part 6. The batch and cost discussion close the session.

## A note on how this runs

The whole demo runs offline by default with a deterministic scripted model standing in for the language model. That is deliberate. The reliability machinery, the schema, the verifier, the repair loop, the cost log, is the lesson, and scripting the model makes the failure-then-repair happen the same way on every run, which a live stochastic model would not guarantee in a class. The same pipeline calls a real local model, LM Studio or Ollama, when you set a backend. No GPU and no downloads for the default path.

## Worked target output

This is the moment we are building toward. For the trap question, the scripted model first returns a wrong answer that trusts an outdated flyer, the verifier catches it, and the repair fixes it. Verified output from this notebook:

```
QUERY: What year was the Timberline 18 volt drill released?
  attempt 1 (generate): "released in 2015 [doc:counterexample_doc#0]"
  verify: FAIL -> Year '2015' is less supported in evidence than '2021' (1 vs 2)
  attempt 2 (repair):   "released in 2021 [doc:drill_spec_official#0]"
  verify: PASS
  final status: VERIFIED  attempts: 2
```

The model got it wrong, the system caught it without a human, and the answer that came out was correct. That is the whole point.

## Part 0: Setup

Offline by default. No GPU, no downloads. The stack is Pydantic for the schema guard, plus pandas and matplotlib for the batch metrics. Counter is imported up front because the year guard uses it later, so no cell surprises us with a missing import.

In [ ]:
%pip install -q pydantic pandas matplotlib

In [ ]:
import os
import re
import json
import math
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel, Field, ValidationError, field_validator

import sys, platform
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
import pydantic
print("pydantic:", pydantic.__version__)
print("pandas:", pd.__version__)

## Part 1: Knowledge base and retrieval

A small Cordwell knowledge base. The Timberline drill release year is the fact under test. It is 2021, and 2021 is stated in two documents, the official spec and a catalog note. One counterexample document, an outdated flyer, wrongly says 2015. That majority-of-two versus minority-of-one is what the verifier will use to catch a wrong answer later.

Retrieval is a recap from Demo 2, here with a BM25 scorer instead of TF-IDF. It is pre-written. The concepts in this demo are schema, verify, and repair, not retrieval, so we do not spend time here.

In [ ]:
DOCUMENTS = {
    "drill_spec_official": [
        "The Timberline 18 volt drill was released in 2021 and delivers 500 inch pounds of torque.",
        "The drill ships with one battery and a charger. [source: product-sheet-TL18V]",
    ],
    "catalog_notes": [
        "The Timberline 18 volt drill has been a catalog staple since 2021.",
    ],
    "returns_policy": [
        "Most items may be returned within 90 days with proof of purchase.",
    ],
    "warranty": [
        "Cordwell branded power tools carry a 3 year limited warranty.",
    ],
    "counterexample_doc": [
        "An outdated flyer wrongly lists the Timberline 18 volt drill release year as 2015.",
    ],
    "abstention_policy": [
        "When the knowledge base lacks a confident answer the assistant declines and routes to a human associate.",
    ],
}

def tokenize(text):
    return re.findall(r"[A-Za-z0-9']+", text.lower())

DOC_TF = {d: [tokenize(x) for x in ch] for d, ch in DOCUMENTS.items()}
DF = defaultdict(int)
for d, ch in DOC_TF.items():
    for toks in ch:
        for t in set(toks):
            DF[t] += 1
N_CHUNKS = sum(len(ch) for ch in DOC_TF.values())

def bm25(qtok, ctok, k1=1.5, b=0.75, avgdl=12.0):
    score, dl = 0.0, len(ctok)
    for q in qtok:
        f = ctok.count(q)
        if f == 0:
            continue
        idf = math.log((N_CHUNKS - DF[q] + 0.5) / (DF[q] + 0.5) + 1.0)
        score += idf * (f * (k1 + 1)) / (f + k1 * (1 - b + b * (dl / avgdl)))
    return score

def retrieve(query, top_k=3):
    qtok = tokenize(query)
    scored = []
    for d, ch in DOCUMENTS.items():
        for i, c in enumerate(ch):
            s = bm25(qtok, tokenize(c))
            if s > 0:
                scored.append((d, i, c, s))
    scored.sort(key=lambda x: x[3], reverse=True)
    return scored[:top_k]

for d, i, t, s in retrieve("What year was the Timberline 18 volt drill released?"):
    print(f"  {d}#{i} score={s:.3f}: {t[:55]}")

## Part 2: Schema guard with Pydantic

The first line of defense is structure. We require the model to return JSON that fits a Pydantic schema: an answer string, a non-empty list of citations, a confidence between 0 and 1, and an abstained flag. Anything that does not parse or does not fit is rejected before any downstream code trusts it. A validator also rejects duplicate citations, a common way models pad an answer to look better supported than it is.

WE-DO here. One line. The schema guard itself, validating the parsed JSON against the schema.

In [ ]:
class Citation(BaseModel):
    doc_id: str = Field(..., min_length=1)
    chunk_id: int = Field(..., ge=0)
    quote: str = Field(..., min_length=1)

class AnswerSchema(BaseModel):
    answer: str = Field(..., min_length=3, max_length=2000)
    citations: List[Citation] = Field(..., min_length=1)
    confidence: float = Field(..., ge=0, le=1)
    abstained: bool = False

    @field_validator("citations")
    @classmethod
    def unique_citations(cls, v):
        seen = set()
        for c in v:
            key = (c.doc_id, c.chunk_id, c.quote.strip())
            if key in seen:
                raise ValueError("Duplicate citation entries detected")
            seen.add(key)
        return v

def validate_json_output(raw):
    try:
        data = json.loads(raw)
    except Exception as e:
        return None, [f"JSON parse error: {e}"]
    try:
        # ===================== WE-DO =====================
        # Schema guard: validate the parsed dict against AnswerSchema. Use
        # AnswerSchema.model_validate(data). On success return (obj, []).
        obj = ...        # replace ... together
        # =================================================
        return obj, []
    except ValidationError as ve:
        return None, [str(ve)]

# A good payload validates, a bad one is rejected with a reason.
good = json.dumps({"answer": "x [doc:returns_policy#0]",
                   "citations": [{"doc_id": "returns_policy", "chunk_id": 0, "quote": "Most items"}],
                   "confidence": 0.8})
bad = json.dumps({"answer": "x", "citations": [], "confidence": 2.0})
print("good ->", validate_json_output(good)[0] is not None)
print("bad  ->", validate_json_output(bad)[1][0][:60], "...")

## Part 3: The heuristic verifier

Structure alone does not make an answer true. The verifier checks the claims against the retrieved evidence with fast, transparent rules. If the answer abstains, there is no factual claim to ground, so it passes trivially. Otherwise we run four checks.

- Quote containment. Every citation's quoted text must actually appear in the chunk it cites. A fabricated quote fails here.
- Inline citation. The answer must carry at least one inline marker in the form bracket doc colon id hash index.
- Numeric consistency. Any three or four digit number in the answer must appear somewhere in the retrieved evidence.
- Majority-year guard. If the answer states a year, and the evidence favors a different year, flag it. This is the check that catches the outdated flyer.

WE-DO here. Two lines. The quote-containment check, and the majority-year comparison that flags the minority year.

In [ ]:
def _year_counts(retrieved):
    pat = re.compile(r"\b(?:18|19|20)\d{2}\b")
    counts = Counter()
    for _d, _i, t, _s in retrieved:
        for y in pat.findall(t):
            counts[y] += 1
    return counts

def heuristic_verify(obj, retrieved):
    if obj.abstained:
        return {"passed": True, "issues": []}      # an abstention makes no factual claim to ground
    issues = []
    by_key = {(d, i): t for (d, i, t, _s) in retrieved}
    all_text = " ".join(t for (_d, _i, t, _s) in retrieved)

    # 1) quote containment
    for c in obj.citations:
        chunk_text = by_key.get((c.doc_id, c.chunk_id), "")
        # ===================== WE-DO =====================
        # Quote containment: if the citation's quote text does not appear in the
        # cited chunk_text, append an issue naming c.doc_id and c.chunk_id.
        if ...:        # replace ... together
            issues.append(f"Quoted evidence not found in cited chunk: {c.doc_id}#{c.chunk_id}")
        # =================================================

    # 2) at least one inline citation marker
    if not re.search(r"\[doc:[A-Za-z0-9_\-]+#\d+\]", obj.answer):
        issues.append("Answer lacks an inline citation marker like [doc:ID#idx]")

    # 3) numeric consistency
    for n in re.findall(r"\b\d{3,4}\b", obj.answer):
        if n not in all_text:
            issues.append(f"Number '{n}' not found in retrieved evidence")

    # 4) majority-year guard
    years = re.findall(r"\b(?:18|19|20)\d{2}\b", obj.answer)
    if years:
        yc = _year_counts(retrieved)
        if yc:
            top_year, top_count = max(yc.items(), key=lambda kv: kv[1])
            for y in years:
                # ===================== WE-DO =====================
                # Majority-year guard: flag a year that is less supported in the
                # evidence than the top year. Compare yc.get(y, 0) to top_count.
                if ...:        # replace ... together
                    issues.append(f"Year '{y}' is less supported in evidence than "
                                  f"'{top_year}' ({yc.get(y, 0)} vs {top_count})")
                # =================================================

    return {"passed": not issues, "issues": issues}

## Part 4: The model, offline by default, LM Studio or Ollama live

The generation step goes through one chat interface. By default it runs offline against a deterministic scripted model, so the whole verify-and-repair story is reproducible with zero dependencies. Set a backend and the same interface calls a real local model instead.

The scripted model is honest about what it is. It returns canned JSON for each query, and for the trap query it returns the wrong 2015 answer on the first pass and the correct 2021 answer when asked to repair. That lets us watch the verifier catch a mistake and the repair fix it, the same way every run.

Backends. offline is the default. lmstudio calls LM Studio on port 1234. ollama calls Ollama on port 11434. Both live backends speak the OpenAI compatible protocol, so they share one call and differ only by port and model tag. The tag is a config variable defaulting to gemma4.

CURRENCY FLAG. Confirm the gemma4 tag the cohort machines actually pulled, for both LM Studio and Ollama, before running live.

This cell is pre-written. The live call is the same pattern the room built in Demo 2, so here we keep the focus on schema, verify, and repair.

In [ ]:
BACKEND_MODE = os.environ.get("BACKEND_MODE", "offline")   # offline | lmstudio | ollama
LOCAL_MODEL = os.environ.get("LOCAL_MODEL", "gemma4")     # CURRENCY FLAG: confirm the pulled tag
BACKENDS = {
    "lmstudio": {"base_url": "http://localhost:1234/v1", "api_key": "lm-studio"},
    "ollama":   {"base_url": "http://localhost:11434/v1", "api_key": "ollama"},
}

# Deterministic scripted model. Canned JSON per query; the trap query returns a
# wrong answer first and the corrected answer on repair.
CANNED = {
    "released": {
        "initial": {"answer": "The Timberline 18 volt drill was released in 2015. [doc:counterexample_doc#0]",
                    "citations": [{"doc_id": "counterexample_doc", "chunk_id": 0,
                                   "quote": "An outdated flyer wrongly lists the Timberline 18 volt drill release year as 2015."}],
                    "confidence": 0.83, "abstained": False},
        "repair": {"answer": "The Timberline 18 volt drill was released in 2021. [doc:drill_spec_official#0]",
                   "citations": [{"doc_id": "drill_spec_official", "chunk_id": 0,
                                  "quote": "The Timberline 18 volt drill was released in 2021 and delivers 500 inch pounds of torque."}],
                   "confidence": 0.88, "abstained": False},
    },
    "torque": {"initial": {"answer": "The Timberline 18 volt drill delivers 500 inch pounds of torque. [doc:drill_spec_official#0]",
                           "citations": [{"doc_id": "drill_spec_official", "chunk_id": 0,
                                          "quote": "The Timberline 18 volt drill was released in 2021 and delivers 500 inch pounds of torque."}],
                           "confidence": 0.9, "abstained": False}},
    "return": {"initial": {"answer": "Most items may be returned within 90 days with proof of purchase. [doc:returns_policy#0]",
                           "citations": [{"doc_id": "returns_policy", "chunk_id": 0,
                                          "quote": "Most items may be returned within 90 days with proof of purchase."}],
                           "confidence": 0.86, "abstained": False}},
    "warranty": {"initial": {"answer": "Cordwell branded power tools carry a 3 year limited warranty. [doc:warranty#0]",
                             "citations": [{"doc_id": "warranty", "chunk_id": 0,
                                            "quote": "Cordwell branded power tools carry a 3 year limited warranty."}],
                             "confidence": 0.84, "abstained": False}},
    "holiday": {"initial": {"answer": "INSUFFICIENT_INFORMATION: the knowledge base does not cover store holiday hours. [doc:abstention_policy#0]",
                            "citations": [{"doc_id": "abstention_policy", "chunk_id": 0,
                                           "quote": "When the knowledge base lacks a confident answer the assistant declines and routes to a human associate."}],
                            "confidence": 0.3, "abstained": True}},
}

def estimate_tokens(s):
    return max(1, math.ceil(len(s) / 4))

@dataclass
class LLMResult:
    text: str
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0

def _match_key(prompt):
    m = re.search(r"User Question:\s*(.+)", prompt)
    question = (m.group(1) if m else prompt).lower()
    for kw in CANNED:
        if kw in question:
            return kw
    return None

def _offline_chat(system, user):
    is_repair = "revise" in system.lower() or "repair" in system.lower()
    key = _match_key(user)
    canned = CANNED.get(key)
    if canned is None:
        payload = {"answer": "INSUFFICIENT_INFORMATION: no matching evidence. [doc:abstention_policy#0]",
                   "citations": [{"doc_id": "abstention_policy", "chunk_id": 0,
                                  "quote": "When the knowledge base lacks a confident answer the assistant declines and routes to a human associate."}],
                   "confidence": 0.2, "abstained": True}
    else:
        payload = canned["repair"] if (is_repair and "repair" in canned) else canned["initial"]
    text = json.dumps(payload)
    pt, ct = estimate_tokens(system + user), estimate_tokens(text)
    return LLMResult(text, pt, ct, pt + ct)

def chat(system, user):
    """Offline scripted model by default, or a real local model when a backend is set."""
    if BACKEND_MODE == "offline":
        return _offline_chat(system, user)
    if BACKEND_MODE not in BACKENDS:
        raise ValueError(f"Unknown BACKEND_MODE {BACKEND_MODE!r}. Use offline, lmstudio, or ollama.")
    from openai import OpenAI
    cfg = BACKENDS[BACKEND_MODE]
    client = OpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"])
    try:
        resp = client.chat.completions.create(
            model=LOCAL_MODEL,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            temperature=0.0,
        )
        text = resp.choices[0].message.content.strip()
    except Exception as e:
        raise RuntimeError(
            f"Backend {BACKEND_MODE} at {cfg['base_url']} is not reachable. Start it and pull "
            f"{LOCAL_MODEL}, or set BACKEND_MODE=offline. Original error: {e}"
        )
    u = resp.usage
    pt = getattr(u, "prompt_tokens", 0) or estimate_tokens(system + user)
    ct = getattr(u, "completion_tokens", 0) or estimate_tokens(text)
    return LLMResult(text, pt, ct, pt + ct)

print(f"BACKEND_MODE={BACKEND_MODE}  LOCAL_MODEL={LOCAL_MODEL}")

## Part 5: The verify-and-repair loop

Now we wire it together. Generate an answer, validate the schema, verify the claims. If verification passes, we are done. If it fails, feed the specific issues into a repair prompt, regenerate, and check again, up to a cap on attempts. We log attempts and tokens at every stage so we can price the loop later.

WE-DO here. One line. The gate that ends the loop when verification passes.

In [ ]:
@dataclass
class RunLog:
    query: str
    attempts: int = 0
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0
    stages: List[Dict[str, Any]] = field(default_factory=list)
    final_status: str = "UNKNOWN"

def _gen_prompt(query, retrieved):
    docs = "\n\n".join(f"[doc:{d}#{i}] {t}" for (d, i, t, _s) in retrieved)
    system = "You are an evidence-first assistant. Answer only with facts grounded in the sources."
    user = ("Answer using ONLY the sources. Return JSON with answer, citations, confidence, abstained.\n\n"
            f"Sources:\n{docs}\n\nUser Question: {query}")
    return system, user

def _repair_prompt(issues, obj, retrieved, query):
    ctx = "\n\n".join(f"[doc:{d}#{i}] {t}" for (d, i, t, _s) in retrieved)
    system = "You revise ONLY unsupported claims to match the sources. Keep supported parts unchanged."
    user = ("Revise the JSON so all claims are supported. Issues:\n"
            + "\n".join(f"- {x}" for x in issues)
            + f"\n\nSources:\n{ctx}\n\nUser Question: {query}\n\nCurrent JSON:\n{obj.model_dump_json(indent=2)}")
    return system, user

def _accumulate(log, res):
    log.attempts += 1
    log.prompt_tokens += res.prompt_tokens
    log.completion_tokens += res.completion_tokens
    log.total_tokens += res.total_tokens

def verify_and_repair(query, max_attempts=3, top_k=3):
    log = RunLog(query=query)
    retrieved = retrieve(query, top_k=top_k)

    system, user = _gen_prompt(query, retrieved)
    res = chat(system, user)
    _accumulate(log, res)
    obj, errs = validate_json_output(res.text)
    log.stages.append({"stage": "generate", "schema_errors": errs})

    attempt = 0
    while attempt < max_attempts:
        attempt += 1
        if obj is None:
            log.final_status = "FAILED_SCHEMA"
            return obj, log
        hv = heuristic_verify(obj, retrieved)
        log.stages.append({"stage": "verify", "passed": hv["passed"], "issues": hv["issues"]})
        # ===================== WE-DO =====================
        # Verification gate: if the heuristic verifier passed, mark the run
        # VERIFIED and return (obj, log) to stop the loop.
        if ...:        # replace ... together
            log.final_status = "VERIFIED"
            return obj, log
        # =================================================
        system, user = _repair_prompt(hv["issues"], obj, retrieved, query)
        res = chat(system, user)
        _accumulate(log, res)
        obj2, _errs2 = validate_json_output(res.text)
        log.stages.append({"stage": "repair"})
        obj = obj2 if obj2 is not None else obj

    log.final_status = "FAILED_TO_VERIFY"
    return obj, log

print("loop ready")

## Part 6: The counterexample trap

This is the centerpiece. Ask for the drill release year. The evidence holds the correct 2021 in two documents and a wrong 2015 in one outdated flyer. The scripted model first trusts the flyer and answers 2015. Watch the verifier catch it with the majority-year guard, then watch the repair fix it to 2021, with no human in the loop.

Predict with the room before running. What will the verifier say about 2015. What should the repaired answer be.

In [ ]:
trap = "What year was the Timberline 18 volt drill released?"
retrieved = retrieve(trap, top_k=3)
print("Year counts in the retrieved evidence:", dict(_year_counts(retrieved)))
print()

obj, log = verify_and_repair(trap, max_attempts=3, top_k=3)
for st in log.stages:
    if st["stage"] == "verify":
        print(f"  verify: {'PASS' if st['passed'] else 'FAIL'}", "" if st["passed"] else "-> " + st["issues"][0])
    else:
        print(f"  {st['stage']}")
print()
print("final status:", log.final_status, " attempts:", log.attempts)
print("final answer:", obj.answer)

What happened. The evidence favors 2021 by two mentions to one. The first answer used 2015 and cited the flyer, so the majority-year guard flagged it. The repair step rewrote the answer to the supported year and re-cited the official spec, and the second verification passed. The model produced a wrong answer and the system corrected it automatically. That is verify-and-repair earning its cost.

## Part 7: Batch run, reliability and cost

Run the whole query set and log what it took. Four questions are answerable and one is out of scope. We record the final status, the number of attempts, the tokens, and an estimated cost. The trap query costs more because it takes a repair. That extra cost buys reliability, and being able to see the trade-off is the point of logging it.

Cost note. Local models on the cohort machines are free, so the local cost is zero. The cloud column uses an illustrative per-token rate so the trade-off is visible. These are teaching placeholders, not a live price sheet.

In [ ]:
# Illustrative only. Confirm real provider prices before quoting them. Local is free.
PRICE_PER_1K = {"gemma4": {"input": 0.0, "output": 0.0},
                "cloud_illustrative": {"input": 0.00015, "output": 0.0006}}

def est_cost(model, pt, ct):
    p = PRICE_PER_1K.get(model, PRICE_PER_1K["cloud_illustrative"])
    return pt / 1000 * p["input"] + ct / 1000 * p["output"]

def should_auto_abstain(obj, min_conf=0.6):
    if obj.abstained:
        return True
    # ===================== WE-DO =====================
    # Abstention on low confidence: if the answer's confidence is below min_conf,
    # return True to force an abstention.
    if ...:        # replace ... together
        return True
    # =================================================
    return False

QUERIES = [
    "What year was the Timberline 18 volt drill released?",
    "How much torque does the Timberline 18 volt drill deliver?",
    "What is the return window for most items?",
    "What warranty do Cordwell power tools carry?",
    "What are the store holiday hours?",
]

rows = []
for q in QUERIES:
    obj, log = verify_and_repair(q, max_attempts=3, top_k=3)
    status = log.final_status
    if obj is not None and should_auto_abstain(obj) and status == "VERIFIED":
        status = "ABSTAINED"
    rows.append({
        "query": q[:42],
        "status": status,
        "attempts": log.attempts,
        "tokens": log.total_tokens,
        "cloud_cost_usd": round(est_cost("cloud_illustrative", log.prompt_tokens, log.completion_tokens), 6),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

Read the table with the room. Four of five reach a trusted state, four VERIFIED and one ABSTAINED, and none passed through an unsupported claim. The trap query took two attempts and more tokens than the rest. That single repair is the price of catching one wrong answer.

In [ ]:
verified = (df["status"] == "VERIFIED").sum()
abstained = (df["status"] == "ABSTAINED").sum()
avg_attempts = df["attempts"].mean()
print(f"verified: {verified}/{len(df)}   abstained: {abstained}   avg attempts: {avg_attempts:.2f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.2))
df["status"].value_counts().plot(kind="bar", ax=ax1, title="Final status")
ax1.set_ylabel("count")
ax2.bar(range(len(df)), df["attempts"])
ax2.set_title("Attempts per query")
ax2.set_xlabel("query index")
ax2.set_ylabel("attempts")
plt.tight_layout()
plt.show()

## Recap

- Structure first. A schema guard rejects malformed output before anything downstream trusts it.
- Verify the claims. Fast heuristic checks, quote containment, inline citations, and numeric and year consistency, catch unsupported statements without a human.
- Repair with feedback. Feeding specific issues back into the model fixes an unfaithful answer automatically, up to a cap.
- Abstain on low confidence. Declining is a first-class outcome, not a failure.
- Reliability has a price. The repair loop costs extra calls and tokens, and logging makes that trade-off visible so you can tune it.

Responsible AI takeaway. Verification is a safety net, not a guarantee. Heuristic checks catch the failures they are written for and miss the ones they are not. A stronger verifier, an LLM fact-checker or a natural language inference model, catches more but costs more, which is the same trade-off in a different place. Layer cheap checks first and reserve expensive ones for what gets through.

Where this goes next. The same loop takes a real LLM verifier in place of the heuristics, self-consistency across several generations, and persistent run logs for production monitoring. Those are deliberately left as extensions so this session stays on the core loop.

## Pre-flight self-check

Run this last, before class. It confirms the schema guard, the verifier, the repair loop, and abstention all behave. In the student notebook it passes only after every WE-DO line is filled in.

In [ ]:
# 1. Schema guard rejects malformed output.
assert validate_json_output('{"answer": "x", "citations": [], "confidence": 2.0}')[0] is None, \
    "Schema guard should reject an out-of-range confidence and empty citations."

# 2. The trap query repairs to the supported year.
obj, log = verify_and_repair("What year was the Timberline 18 volt drill released?", max_attempts=3)
assert log.final_status == "VERIFIED", "Trap query should end VERIFIED after repair."
assert "2021" in obj.answer and "2015" not in obj.answer, "Repaired answer should use the supported year 2021."
assert log.attempts >= 2, "Trap query should take at least one repair."

# 3. A clean query verifies on the first attempt.
_o, clean = verify_and_repair("What is the return window for most items?", max_attempts=3)
assert clean.final_status == "VERIFIED" and clean.attempts == 1, "Clean query should verify without repair."

# 4. The out-of-scope query abstains.
o3, l3 = verify_and_repair("What are the store holiday hours?", max_attempts=3)
assert o3 is not None and should_auto_abstain(o3), "Out-of-scope query should abstain."

print("Pre-flight checks passed.")
print("Schema guard rejects, trap repairs to 2021, clean verifies in one, out-of-scope abstains.")